# 🎮 Steam Game Recommendation Engine

A content-based recommender that suggests similar Steam games using **tag-based cosine similarity**.

**Pipeline:**
1. Load and inspect Steam metadata, descriptions, and tags
2. Explore genre/tag overlap
3. Build a unified tag feature matrix
4. Normalize tag vectors and compute cosine similarity
5. Compare games, search the catalog, and recommend similar titles

**Data files required** (place in the same directory as this notebook):
- `steam_games_clean.csv`
- `steam_descriptions_clean.csv`
- `steam_tags_clean.csv`


## 1. Setup

Import the core libraries used throughout the notebook.

In [ ]:
import numpy as np
import pandas as pd

## 2. Load the Data

Read in the three cleaned Steam datasets: game metadata, descriptions, and tags.

In [ ]:
games_df = pd.read_csv('steam_games_clean.csv')
descriptions_df = pd.read_csv('steam_descriptions_clean.csv')
tags_df = pd.read_csv('steam_tags_clean.csv')

Quick sanity check on dataset sizes:

In [ ]:
print(games_df.shape)
print(descriptions_df.shape)
print(tags_df.shape)

### 2.1 Check ID Overlap

Before merging datasets, confirm how well the `appid` keys line up across tables.

In [ ]:
games_ids = set(games_df["appid"])
tags_ids = set(tags_df["appid"])
description_ids = set(descriptions_df["steam_appid"])

print("Games + Tags overlap:", len(games_ids & tags_ids))
print("Games without tags:", len(games_ids - tags_ids))
print("Tags without game metadata:", len(tags_ids - games_ids))

print("\nGames + Descriptions overlap:", len(games_ids & description_ids))
print("Games without descriptions:", len(games_ids - description_ids))

## 3. Explore Genres vs. Tags

Check whether the `genres` column in the games dataset is already captured by the tag columns, or whether it adds new information.

In [ ]:
# Get unique genres from the games dataset

unique_genres = set()

for genres in games_df["genres"].dropna():
    for genre in genres.split(";"):
        unique_genres.add(genre.strip().lower().replace(" ", "_"))

print("Unique genres:")
print(sorted(unique_genres))

print("\nNumber of genres:", len(unique_genres))

# Compare with tag columns
tag_columns = set(tags_df.columns.drop("appid"))

genres_already_in_tags = unique_genres & tag_columns
genres_not_in_tags = unique_genres - tag_columns

print("\nGenres already represented as tags:")
print(sorted(genres_already_in_tags))

print("\nGenres NOT represented as tags:")
print(sorted(genres_not_in_tags))

## 4. Build the Model Dataset

Merge game names with their tag vectors to create the dataset used for similarity modeling.

In [ ]:
model_df = games_df[["appid", "name"]].merge(
    tags_df,
    on="appid",
    how="inner"
)

print("Model dataset shape:", model_df.shape)

print("\nFirst few games:")
print(
    model_df[["appid", "name"]].head()
)

Preview the full merged dataset:

In [ ]:
model_df

### 4.1 Inspect a Game's Top Tags

Helper to show the strongest tags for a given game (useful for sanity-checking the feature vectors).

In [ ]:
tag_columns = [col for col in model_df.columns if col not in ["appid", "name"]]

def show_top_tags(game_name, top_n=15):

    matches = model_df[
        model_df["name"].str.lower() == game_name.lower()
    ]

    if matches.empty:
        print(f"Game not found: {game_name}")
        return

    game = matches.iloc[0]

    top_tags = (
        game[tag_columns]
        .sort_values(ascending=False)
        .head(top_n)
    )

    print(f"\n{'=' * 60}")
    print(game["name"])
    print(f"App ID: {game['appid']}")
    print(f"{'=' * 60}")

    print(top_tags[top_tags > 0])


show_top_tags("Counter-Strike")
show_top_tags("Half-Life")
show_top_tags("Portal 2")

## 5. Build Feature Matrix & Normalize

Extract the tag columns as a feature matrix and L2-normalize each game's vector so cosine similarity can be computed efficiently via dot products.

In [ ]:
from sklearn.preprocessing import normalize

# Extract only the tag features
tag_columns = [
    col for col in model_df.columns
    if col not in ["appid", "name"]
]

X = model_df[tag_columns]

# Normalize each game's tag vector to length 1
X_normalized = normalize(X, norm="l2")

print("Original shape:", X.shape)
print("Normalized shape:", X_normalized.shape)

Verify normalization worked — every vector should have length ≈ 1:

In [ ]:
import numpy as np

vector_lengths = np.linalg.norm(X_normalized, axis=1)

print(vector_lengths[:10])

## 6. Compare Two Games

Compute cosine similarity between any two named games.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def get_game_index(game_name):
    matches = model_df[
        model_df["name"].str.lower() == game_name.lower()
    ]

    if matches.empty:
        print(f"Game not found: {game_name}")
        return None

    return matches.index[0]


def compare_games(game1, game2):

    idx1 = get_game_index(game1)
    idx2 = get_game_index(game2)

    if idx1 is None or idx2 is None:
        return

    similarity = cosine_similarity(
        X_normalized[idx1].reshape(1, -1),
        X_normalized[idx2].reshape(1, -1)
    )[0][0]

    print(f"{game1} vs {game2}")
    print(f"Cosine similarity: {similarity:.4f}")

Try it out on a few classic FPS titles:

In [ ]:
compare_games("Counter-Strike", "Team Fortress Classic")
compare_games("Counter-Strike", "Day of Defeat")
compare_games("Counter-Strike", "Portal 2")

## 7. Search the Catalog

Simple substring search over game names, useful for finding the exact title string to pass into the other functions.

In [ ]:
def search_games(query, max_results=10):
    matches = model_df[
        model_df["name"]
        .str.contains(query, case=False, na=False)
    ][["name", "appid"]].head(max_results)

    if matches.empty:
        print(f"No games found matching: '{query}'")
    else:
        print(f"Games matching '{query}':")
        display(matches)

Sanity-check similarity scores on a set of metroidvania-style games:

In [ ]:
compare_games("Hollow Knight", "Ori and the Blind Forest: Definitive Edition")

compare_games("Hollow Knight", "Dead Cells")

compare_games("Hollow Knight", "Guacamelee! Gold Edition")

compare_games("Hollow Knight", "Counter-Strike")

## 8. Recommend Similar Games

Given a game, rank every other game in the catalog by cosine similarity and return the top N matches.

In [ ]:
import numpy as np

def find_similar_games(game_name, top_n=10):

    # Find the game
    matches = model_df[
        model_df["name"].str.lower() == game_name.lower()
    ]

    if matches.empty:
        print(f"Game not found: {game_name}")
        return

    game_index = matches.index[0]

    # Calculate cosine similarity against every game
    similarities = X_normalized @ X_normalized[game_index]

    # Get indices sorted by similarity
    similar_indices = np.argsort(similarities)[::-1]

    # Remove the game itself and keep top N
    similar_indices = [
        idx for idx in similar_indices
        if idx != game_index
    ][:top_n]

    # Create results
    results = model_df.iloc[similar_indices][["name", "appid"]].copy()
    results["similarity"] = similarities[similar_indices]

    return results

## 9. Demo

Search for a game, then find its closest recommendations.

In [ ]:
search_games("Messenger")

In [ ]:
find_similar_games("The Messenger", top_n=10)

In [ ]:
model_df.head()

In [ ]:
model_df.columns

In [ ]:
model_df.to_pickle("model_df.pkl")

In [ ]:
import numpy as np

np.save(
    "data/processed/tag_matrix_normalized.npy",
    X_normalized
)

print("Saved:", X_normalized.shape)